# Read PySpark Config from Files

This notebook demonstrates three ways to load external configuration:

1. **jproperties** — Java-style `.properties` files
2. **ConfigParser (Basic)** — INI-style with `%(key)s` interpolation
3. **ConfigParser (Extended)** — INI-style with `${section:key}` interpolation

In [ ]:
import os

from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("read-config-notebook")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.ui.enabled", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)

## 1. jproperties

Read a Java-style `.properties` file and apply values to the Spark session.

In [ ]:
from jproperties import Properties

configs = Properties()

with open("../cfg/config.properties", "rb") as f:
    configs.load(f)

print("Properties loaded:")
for key, value in configs.items():
    print(f"  {key} = {value.data}")

In [ ]:
# Apply a property to the running session
partitions = configs.get("spark.sql.shuffle.partitions")
if partitions:
    spark.conf.set("spark.sql.shuffle.partitions", partitions.data)
    print("shuffle.partitions =", spark.conf.get("spark.sql.shuffle.partitions"))

## 2. ConfigParser — BasicInterpolation

Uses `%(key)s` syntax for variable substitution.

In [ ]:
from configparser import ConfigParser

basic = ConfigParser()
basic.read("../cfg/config.cfg")

print("config.cfg (BasicInterpolation):")
print("  a =", basic.get("data", "a"))
print("  b =", basic.get("data", "b"))
print("  c =", basic.get("data", "c"))

## 3. ConfigParser — ExtendedInterpolation

Uses `${section:key}` syntax for cross-section references.

In [ ]:
from configparser import ExtendedInterpolation

extended = ConfigParser(interpolation=ExtendedInterpolation())
extended.read("../cfg/config.conf")

print("config.conf (ExtendedInterpolation):")
print("  a =", extended.get("data", "a"))
print("  b =", extended.get("data", "b"))
print("  c =", extended.get("data", "c"))

In [ ]:
spark.stop()